# A2.7 · Attribution: an audit trail that answers "who"

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.6 · Ingress: marking untrusted content at the door](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry, Sigstore |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

A trace that records tool calls is not evidence. Evidence answers who asked, which agent acted, what authority it held, and what input made it act — and an auditor will ask all four in that order.

## 2 · The framework

```
   the four fields an auditor asks for, in order

   1. principal      dana@corp          who asked
   2. agent          patch-agent#7      who acted
   3. authority      repo:write, exp+90s   under what
   4. motivating input   PR #412 body, [data]   why

   a trace with 1-3 and not 4 cannot distinguish authorised from injected
```

**Mitigates: T8 Repudiation & Untraceability · T13 Rogue Agents.**

A1.13 showed a log that was complete for debugging and empty for investigation.
This is the record that is not.

Four fields, each answering a question the tool-call log could not:

**The human principal** — who caused this. From A2.1.

**The agent identity and instance** — what performed it, and which run. From
A2.2, so it is attested rather than claimed.

**The delegation chain** — how authority got from the human to this action. From
A2.3, which is also what makes A1.16's laundering path visible.

**The motivating input, with its origin** — what made the agent decide. From
A2.6. This is the field that establishes root cause, and the one most often
missing, because logging tool calls feels like logging decisions.

Then the structural property, which is not a field: **the store must be outside
the agent's reach.** An agent with broad credentials can usually touch the
logging stack, and a record the actor can edit is not evidence. Append-only,
different credential, ideally different trust domain.

The test is not whether the log looks thorough. It is whether you can answer
"which user caused this, and what made it happen" without asking anyone.

> **What this control closes.**
>
> Makes the incident answerable. Also the control an auditor asks for first, because a system that cannot attribute an action cannot be defended even on a quiet day.

## 3 · The control

In [ ]:
LEDGER = []          # append-only, and the agent holds no credential for it

def record(principal, workload, instance, chain, action, motivating):
    LEDGER.append({
        "principal": principal, "workload": workload, "instance": instance,
        "chain": list(chain), "action": action,
        "motivating_input": motivating["text"][:44],
        "input_origin": motivating["origin"],
    })

def agent_writes(entry):
    """The agent tries to amend the record."""
    raise PermissionError("ledger is append-only and out of the agent's trust domain")

record("dana@corp", "spiffe://corp/reports-agent", "run-8812",
       ["dana@corp", "orchestrator", "reports-agent"],
       "run_query DELETE FROM invoices WHERE id=8812",
       {"text": "wiki/473: retire invoice 8812 when the customer closes", "origin": "knowledge"})

QUESTIONS = {
 "which user caused the deletion?":         lambda e: e["principal"],
 "what performed it?":                      lambda e: f"{e['workload']} ({e['instance']})",
 "how did authority reach it?":             lambda e: " -> ".join(e["chain"]),
 "what made the agent decide?":             lambda e: f"{e['motivating_input']!r} from {e['input_origin']}",
}
e = LEDGER[0]
for q, answer in QUESTIONS.items():
    print(f"   {q:36s}{answer(e)}")

print(f"\nquestions answerable: {len(QUESTIONS)}/{len(QUESTIONS)}")
print()
print(f"input origin was {e['input_origin']!r} - a trust-0 component. That single")
print("field turns 'the agent deleted an invoice' into 'a wiki page told it to',")
print("which is a root cause rather than an observation.")

try:
    agent_writes({"action": "tidy up"})
except PermissionError as err:
    print(f"\nagent attempting to amend the ledger: refused ({err})")
assert len(LEDGER) == 1 and LEDGER[0]["input_origin"] == "knowledge"

## What you just proved

One ledger entry answers all four investigation questions — the human principal, the attested workload and run, the delegation chain, and the motivating input with its origin — and the agent's attempt to amend the record is refused.

## Your turn

Take the last significant action one of your agents performed and try to fill in these four fields from what you actually logged. The missing one is almost always the motivating input.

---

**Next → [A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*